 # Feature Engineering - YouTube Retention Project

In [1]:
import pandas as pd
df = pd.read_csv("../data/processed/cleaned_videos.csv")
df.head()

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description
0,2kyS6SvSYSE,17.14.11,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,2017-11-13 17:13:01+00:00,SHANtell martin,748374,57527,2966,15954,https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg,False,False,False,SHANTELL'S CHANNEL - https://www.youtube.com/s...
1,1ZAPwfrtAFY,17.14.11,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,2017-11-13 07:30:00+00:00,"last week tonight trump presidency|""last week ...",2418783,97185,6146,12703,https://i.ytimg.com/vi/1ZAPwfrtAFY/default.jpg,False,False,False,"One year after the presidential election, John..."
2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12 19:05:24+00:00,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146033,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...
3,puqaWrEC7tY,17.14.11,Nickelback Lyrics: Real or Fake?,Good Mythical Morning,24,2017-11-13 11:00:04+00:00,"rhett and link|""gmm""|""good mythical morning""|""...",343168,10172,666,2146,https://i.ytimg.com/vi/puqaWrEC7tY/default.jpg,False,False,False,Today we find out if Link is a Nickelback amat...
4,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12 18:01:41+00:00,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095731,132235,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...


## Objective
Create meaningful features that help identify high-retention content.

In [2]:
df['engagement_rate'] = (df['likes'] + df['comment_count']) / df['views']

df['like_ratio'] = df['likes'] / df['views']
df['comment_ratio'] = df['comment_count'] / df['views']

In [3]:
df['publish_time'] = pd.to_datetime(df['publish_time'])

df['day'] = df['publish_time'].dt.day_name()
df['hour'] = df['publish_time'].dt.hour

In [4]:
from textblob import TextBlob

def get_sentiment(text):
    try:
        return TextBlob(str(text)).sentiment.polarity
    except:
        return 0

df['sentiment_score'] = df['description'].apply(get_sentiment)

In [5]:
df['controversy'] = df['dislikes'] / df['views']

## Features Created
- Engagement Rate (likes + comments / views)
- Like Ratio
- Comment Ratio
- Day and Hour of publishing
- Sentiment Score (from description)
- Controversy Score (dislikes / views)

In [6]:
df[['engagement_rate', 'like_ratio', 'comment_ratio', 'sentiment_score']].describe()

,engagement_rate,like_ratio,comment_ratio,sentiment_score
count,39907.000000,39907.000000,39907.000000,39907.000000
mean,0.039231,0.034754,0.004477,0.185492
std,0.030037,0.027120,0.005757,0.198739
min,0.000000,0.000000,0.000000,-1.000000
25%,0.018004,0.015279,0.001623,0.047426
50%,0.032449,0.028622,0.002977,0.183418
75%,0.052366,0.047160,0.005245,0.308333
max,0.325928,0.290466,0.117643,1.000000


## Key Insight
Raw metrics like views are not enough to determine content quality. 
Combining engagement, sentiment, and timing provides a more complete understanding of video performance.

In [7]:
df.to_csv("../data/processed/featured_videos.csv", index=False)

# Scoring Model - YouTube Retention Project

## Objective
Combine multiple features into a single score to rank videos based on retention potential.

In [8]:
import sys
sys.path.append("../src")

from scoring_model import scoring_pipeline


In [9]:
df = scoring_pipeline(df)

## Approach
- Normalized features to ensure fair comparison
- Assigned weights to balance popularity, engagement, sentiment, and timing
- Computed a final score for each video

In [10]:
df[['views_n', 'engagement_n', 'sentiment_n', 'time_score', 'score']].head()

,views_n,engagement_n,sentiment_n,time_score,score
0,0.024975,0.301255,0.416667,0.5,0.247514
1,0.080761,0.139390,0.539583,0.5,0.234970
2,0.106565,0.148257,0.572917,1.0,0.303771
3,0.011442,0.110132,0.491335,0.5,0.190818
4,0.069972,0.219239,0.729545,1.0,0.347133


## Key Insights
- High engagement videos rank higher even if views are moderate
- Videos published during peak hours receive a boost
- Sentiment helps prioritize positive user experience
- The model avoids over-prioritizing purely viral content

In [11]:
df.sort_values(by='score', ascending=False).head(10)

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,...,comment_ratio,day,hour,sentiment_score,controversy,views_n,engagement_n,sentiment_n,time_score,score
34257,p8npDG2ulKQ,18.16.05,BTS (방탄소년단) LOVE YOURSELF 轉 Tear 'Singularity'...,ibighit,10,2018-05-06 15:00:02+00:00,"BIGHIT|""빅히트""|""방탄소년단""|""BTS""|""BANGTAN""|""방탄""",29741771,2700800,29341,...,0.012503,Sunday,15,0.350000,0.000987,0.993261,0.316976,0.675000,0.5,0.643583
15226,oWjxSkJpxFU,18.01.02,Suicide: Be Here Tomorrow.,Logan Paul Vlogs,29,2018-01-24 18:30:01+00:00,"logan paul vlog|""logan paul""|""logan""|""paul""|""l...",24286474,1988746,497847,...,0.027099,Wednesday,18,0.333333,0.020499,0.811072,0.334385,0.666667,1.0,0.634243
14999,oWjxSkJpxFU,18.31.01,Suicide: Be Here Tomorrow.,Logan Paul Vlogs,29,2018-01-24 18:30:01+00:00,"logan paul vlog|""logan paul""|""logan""|""paul""|""l...",23742391,1970266,487820,...,0.027273,Wednesday,18,0.333333,0.020546,0.792901,0.338290,0.666667,1.0,0.629250
14785,oWjxSkJpxFU,18.30.01,Suicide: Be Here Tomorrow.,Logan Paul Vlogs,29,2018-01-24 18:30:01+00:00,"logan paul vlog|""logan paul""|""logan""|""paul""|""l...",23164093,1949284,476337,...,0.027461,Wednesday,18,0.333333,0.020564,0.773588,0.342443,0.666667,1.0,0.623944
33804,p8npDG2ulKQ,18.14.05,BTS (방탄소년단) LOVE YOURSELF 轉 Tear 'Singularity'...,ibighit,10,2018-05-06 15:00:02+00:00,"BIGHIT|""빅히트""|""방탄소년단""|""BTS""|""BANGTAN""|""방탄""",26912663,2636004,27675,...,0.013633,Sunday,15,0.350000,0.001028,0.898778,0.342344,0.675000,0.5,0.619393
14535,oWjxSkJpxFU,18.29.01,Suicide: Be Here Tomorrow.,Logan Paul Vlogs,29,2018-01-24 18:30:01+00:00,"logan paul vlog|""logan paul""|""logan""|""paul""|""l...",22387656,1919981,461659,...,0.027917,Wednesday,18,0.333333,0.020621,0.747657,0.348782,0.666667,1.0,0.617087
33690,p8npDG2ulKQ,18.13.05,BTS (방탄소년단) LOVE YOURSELF 轉 Tear 'Singularity'...,ibighit,10,2018-05-06 15:00:02+00:00,"BIGHIT|""빅히트""|""방탄소년단""|""BTS""|""BANGTAN""|""방탄""",25924568,2613905,27016,...,0.014017,Sunday,15,0.350000,0.001042,0.865779,0.352360,0.675000,0.5,0.611349
14309,oWjxSkJpxFU,18.28.01,Suicide: Be Here Tomorrow.,Logan Paul Vlogs,29,2018-01-24 18:30:01+00:00,"logan paul vlog|""logan paul""|""logan""|""paul""|""l...",21104436,1867610,435189,...,0.028792,Wednesday,18,0.333333,0.020621,0.704802,0.359853,0.666667,1.0,0.605962
8734,l_lblj8Cq0o,17.28.12,"G-Eazy - No Limit REMIX ft. A$AP Rocky, Cardi ...",GEazyMusicVEVO,10,2017-12-19 20:00:05+00:00,"BPG/RVG/RCA Records|""G-Eazy feat. A$AP Rocky""|...",28474364,588244,31954,...,0.001335,Tuesday,20,0.471591,0.001122,0.950934,0.067480,0.735795,1.0,0.603604
33480,p8npDG2ulKQ,18.12.05,BTS (방탄소년단) LOVE YOURSELF 轉 Tear 'Singularity'...,ibighit,10,2018-05-06 15:00:02+00:00,"BIGHIT|""빅히트""|""방탄소년단""|""BTS""|""BANGTAN""|""방탄""",24186625,2571889,25690,...,0.014797,Sunday,15,0.350000,0.001062,0.807737,0.371653,0.675000,0.5,0.597786


## Conclusion
A weighted scoring model provides a more balanced and realistic ranking compared to using a single metric like views. This approach better aligns with user retention and watch-time optimization goals.

In [12]:
df.to_csv("../data/processed/scored_videos.csv", index=False)